# LeetCode #1055: Shortest Way to Form String

https://leetcode.com/problems/shortest-way-to-form-string/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(|source| \times |target|)$ | $O(1)$ |
| **Optimal: Next-Character Index Table ★** | $O(|source| + |target| \times 26)$ | $O(|source| \times 26)$ |

---

## Understanding the Methods

### Brute Force
Greedily scan `source` from left to right for each character of `target`, restarting `source` when exhausted. Each restart costs $O(|source|)$, giving $O(|source| \times |target|)$ overall.

### Optimal: Next-Character Index Table ★
Precompute a table `nxt[i][c]` = the smallest index $\geq i$ in `source` where character `c` appears (or `-1` if absent). Each target character lookup then jumps to the right position in $O(1)$, reducing total work to $O(|target|)$ after an $O(|source| \times 26)$ build step.

**Constraints:**
* $1 \leq source.length, target.length \leq 1000$
* `source` and `target` consist of lowercase English letters.

## Solutions

### C#

In [ ]:
public class Solution {
    public int ShortestWay(string source, string target) {
        int m = source.Length;
        // nxt[i][c] = first occurrence of char c at or after position i in source; -1 if none
        var nxt = new int[m + 1, 26];
        for (int c = 0; c < 26; c++) nxt[m, c] = -1;
        for (int i = m - 1; i >= 0; i--) {
            for (int c = 0; c < 26; c++) nxt[i, c] = nxt[i + 1, c];
            nxt[i, source[i] - 'a'] = i;
        }
        int copies = 1, pos = 0;
        foreach (char ch in target) {
            int c = ch - 'a';
            if (nxt[0, c] == -1) return -1; // character not in source at all
            if (nxt[pos, c] == -1) {
                // Exhausted source without finding ch — need another copy
                copies++;
                pos = 0;
            }
            // Jump directly to the next occurrence and advance past it
            pos = nxt[pos, c] + 1;
        }
        return copies;
    }
}

### Python

In [ ]:
class Solution:
    def shortest_way(self, source: str, target: str) -> int:
        m = len(source)
        # nxt[i][c] = first occurrence of char c at or after position i in source; -1 if none
        nxt = [[-1] * 26 for _ in range(m + 1)]
        for i in range(m - 1, -1, -1):
            nxt[i] = nxt[i + 1][:]
            nxt[i][ord(source[i]) - ord('a')] = i
        copies, pos = 1, 0
        for ch in target:
            c = ord(ch) - ord('a')
            if nxt[0][c] == -1:
                return -1  # character not in source at all
            if nxt[pos][c] == -1:
                # Exhausted source without finding ch — need another copy
                copies += 1
                pos = 0
            # Jump directly to the next occurrence and advance past it
            pos = nxt[pos][c] + 1
        return copies

### Go

In [ ]:
func shortestWay(source string, target string) int {
    m := len(source)
    // nxt[i][c] = first occurrence of char c at or after position i in source; -1 if none
    nxt := make([][26]int, m+1)
    for c := 0; c < 26; c++ { nxt[m][c] = -1 }
    for i := m - 1; i >= 0; i-- {
        nxt[i] = nxt[i+1]
        nxt[i][source[i]-'a'] = i
    }
    copies, pos := 1, 0
    for _, ch := range target {
        c := int(ch - 'a')
        if nxt[0][c] == -1 { return -1 } // character not in source at all
        if nxt[pos][c] == -1 {
            // Exhausted source without finding ch — need another copy
            copies++
            pos = 0
        }
        // Jump directly to the next occurrence and advance past it
        pos = nxt[pos][c] + 1
    }
    return copies
}

### Rust

In [ ]:
impl Solution {
    pub fn shortest_way(source: String, target: String) -> i32 {
        let src: Vec<usize> = source.bytes().map(|b| (b - b'a') as usize).collect();
        let m = src.len();
        // nxt[i][c] = first occurrence of char c at or after position i in source; -1 if none
        let mut nxt = vec![[-1i32; 26]; m + 1];
        for i in (0..m).rev() {
            nxt[i] = nxt[i + 1];
            nxt[i][src[i]] = i as i32;
        }
        let mut copies = 1i32;
        let mut pos = 0usize;
        for b in target.bytes() {
            let c = (b - b'a') as usize;
            if nxt[0][c] == -1 { return -1; } // character not in source at all
            if nxt[pos][c] == -1 {
                // Exhausted source without finding ch — need another copy
                copies += 1;
                pos = 0;
            }
            // Jump directly to the next occurrence and advance past it
            pos = nxt[pos][c] as usize + 1;
        }
        copies
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `source = "abc"`, `target = "abcbc"`
One pass covers `abc`. At `b`, position wraps to a second copy, covering `bc`. Total copies: **2**.

### 2. Slightly Complex
**Input:** `source = "xyz"`, `target = "xzy"`
After matching `x` and `z`, the next target char `y` appears after `z` in source at index 2, so `pos = 3`, wrapping to copy 2 to match `y`. Answer: **2**.

### 3. Edge Case: Time Factor
**Input:** `source = "a"`, `target = "aaa...a"` (1000 `a`s).
Each character triggers a source-exhaustion wrap (since $m = 1$). The algorithm performs 1000 table lookups — all $O(1)$ — but increments `copies` 1000 times. Answer: **1000**.

### 4. Edge Case: Space Factor
**Input:** `source` of length 1000 (all distinct letters repeated cyclically).
The `nxt` table has $(1000 + 1) \times 26 = 26{,}026$ entries — the maximum $O(|source| \times 26)$ footprint. Build cost dominates at $O(26{,}000)$ assignments.

### 5. Almost-Impossible but Plausible
**Input:** `source = "ab"`, `target = "ba"`.
The table lookup for `b` at `pos=0` gives index 1; then `a` at `pos=2` wraps to copy 2 and matches index 0. Result: **2** — even though both characters exist in source, the ordering mismatch forces two copies.